# Road Accident Severity Classification

## 1. Introduction

This project aims to develop a data-driven approach to predict whether a road accident results in a severe injury. Using the French BAAC (Bulletin d’Analyse des Accidents Corporels) dataset, we construct a structured analytical dataset combining environmental, user-level, and vehicle-level features.

A baseline classification model is implemented to estimate the probability of severe outcomes. The analysis focuses not only on predictive performance, but also on understanding how different factors contribute to accident severity. Particular attention is given to evaluation metrics and threshold selection, highlighting the trade-offs between detecting severe cases and limiting false alarms.

In [288]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

## 2. Data Description & Preprocessing  

### 2.1 Data Source (BAAC Dataset)

In [289]:
df1 = pd.read_csv("caract-2024.csv", sep=";", engine="python")
df2 = pd.read_csv("usagers-2024.csv", sep=";", engine="python")
df3 = pd.read_csv("vehicules-2024.csv", sep=";", engine="python")


### 2.2 Data Cleaning

In [290]:
df1.columns

Index(['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
       'int', 'atm', 'col', 'adr', 'lat', 'long'],
      dtype='object')

In [291]:
situation = df1[
   [ "Num_Acc", "mois", "jour",
   "hrmn", "lum", "atm", "int","adr"]
].copy()
situation.head()

,Num_Acc,mois,jour,hrmn,lum,atm,int,adr
0,202400000001,3,25,07:40,2,5,1,D438
1,202400000002,3,20,15:05,1,7,3,HOTEL DIEU (RUE DE L')
2,202400000003,3,22,19:30,2,1,1,Allée des Tilleuls
3,202400000004,3,24,17:50,1,7,3,128 Rue d'Authie
4,202400000005,3,25,19:35,5,2,3,BEDOULE (CHEMIN DE LA)


In [292]:
df2.columns

Index(['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu',
       'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp',
       'actp', 'etatp'],
      dtype='object')

In [293]:
usagers = df2[[
    'Num_Acc','catu',"id_vehicule",'id_usager',
    'grav', 'an_nais','sexe', 'secu1', 'secu2', 'secu3'
]].copy()
usagers.head()

,Num_Acc,catu,id_vehicule,id_usager,grav,an_nais,sexe,secu1,secu2,secu3
0,202400000001,1,155 781 758,203 988 581,3,2003.0,1,1,-1,-1
1,202400000001,1,155 781 759,203 988 582,1,1997.0,1,1,-1,-1
2,202400000002,3,155 781 757,203 988 579,3,1927.0,2,0,-1,-1
3,202400000002,1,155 781 757,203 988 580,1,1987.0,1,1,0,-1
4,202400000003,2,155 781 756,203 988 574,4,2007.0,2,8,0,-1


In [294]:
df3.columns

Index(['Num_Acc', 'id_vehicule', 'num_veh', 'senc', 'catv', 'obs', 'obsm',
       'choc', 'manv', 'motor', 'occutc'],
      dtype='object')

In [295]:
vehc_type = df3[[
    'Num_Acc', 'id_vehicule', 'senc', 
      'catv', 'choc', 'manv'
]].copy()
vehc_type.head()

,Num_Acc,id_vehicule,senc,catv,choc,manv
0,202400000001,155 781 758,1,7,1,13
1,202400000001,155 781 759,2,14,2,21
2,202400000002,155 781 757,1,10,3,15
3,202400000003,155 781 756,3,3,1,1
4,202400000004,155 781 754,3,34,0,0


In [296]:
situation.dropna()
usagers.dropna()
vehc_type.dropna()

,Num_Acc,id_vehicule,senc,catv,choc,manv
0,202400000001,155 781 758,1,7,1,13
1,202400000001,155 781 759,2,14,2,21
2,202400000002,155 781 757,1,10,3,15
3,202400000003,155 781 756,3,3,1,1
4,202400000004,155 781 754,3,34,0,0
...,...,...,...,...,...,...
92673,202400054401,155 686 119,1,60,7,0
92674,202400054401,155 686 120,1,33,1,1
92675,202400054402,155 686 118,1,10,1,2
92676,202400054402,155 686 121,1,7,4,2


### 2.3 Data Integration

In [297]:
data_m = situation
data_m = data_m.merge(
usagers,
on="Num_Acc",
how="left"
)
data_m = data_m.merge(
    vehc_type,
    on=["Num_Acc","id_vehicule"],
    how="left"
)
data_m.columns

Index(['Num_Acc', 'mois', 'jour', 'hrmn', 'lum', 'atm', 'int', 'adr', 'catu',
       'id_vehicule', 'id_usager', 'grav', 'an_nais', 'sexe', 'secu1', 'secu2',
       'secu3', 'senc', 'catv', 'choc', 'manv'],
      dtype='object')

### 2.4 Data Reorganization and Feature Construction

In [298]:
data = data_m.rename(columns={
    "Num_Acc": "accident_id",
    "mois": "month",
    "jour": "day",
    "hrmn": "time",
    "lum": "light",
    "atm": "weather",
    "int": "intersection",
    "adr": "road",
    "catu": "user_type",
    "id_vehicule": "vehicle_id",
    "id_usager": "user_id",
    "grav": "severity",
    "an_nais": "birth_year",
    "sexe": "sex",
    "senc": "direction",
    "catv": "vehicle_type",
    "choc": "impact",
    "manv": "maneuver"
})

In [299]:
data.columns

Index(['accident_id', 'month', 'day', 'time', 'light', 'weather',
       'intersection', 'road', 'user_type', 'vehicle_id', 'user_id',
       'severity', 'birth_year', 'sex', 'secu1', 'secu2', 'secu3', 'direction',
       'vehicle_type', 'impact', 'maneuver'],
      dtype='object')

#### 2.4.1 Accident-Level Data (Situation)

In [300]:
def group_light_detailed(x):
    if pd.isna(x):
        return "unknown"
    if x == 1:
        return "day"
    if x == 2:
        return "twilight"
    if x == 3:
        return "night_no_light"
    if x in [4, 5]:
        return "night_with_light"
    return "unknown"

data["light"] = data["light"].apply(group_light_detailed)

**Note:** 

The variable `light` (originally `lum`) follows the official BAAC (Bulletin d’Analyse des Accidents Corporels) classification provided by ONISR (France).  

It describes lighting conditions at the time of the accident, with categories including daytime, twilight, and various nighttime conditions depending on public lighting.  
In this project, these categories are grouped into broader levels (day, twilight, night) to reflect differences in visibility.

In [301]:
def group_weather(x):
    if pd.isna(x) or x == -1:
        return "unknown"
    if x == 1:
        return "clear"
    if x in [2, 3]:
        return "rain"
    if x == 4:
        return "snow_hail"
    if x == 5:
        return "fog_smoke"
    if x in [6, 7]:
        return "wind_storm"
    return "other"

data["weather"]= data["weather"].apply(group_weather)

**Note:** 

The variable `weather` (originally `atm`) follows the official BAAC classification provided.  

It describes atmospheric conditions at the time of the accident, including clear weather, rain, snow, fog, and wind.  
For modeling purposes, these categories are grouped into broader classes (e.g., clear, rain, adverse, unknown) to improve interpretability and model stability.

In [302]:
def group_intersection(x):
    if pd.isna(x) or x == -1:
        return "unknown"
    if x == 1:
        return "no_intersection"
    if x in [2, 3, 4, 5]:
        return "standard_intersection"
    if x == 6:
        return "roundabout"
    return "other"

data["intersection"] = data["intersection"].apply(group_intersection)

**Note:** 

The variable `intersection` (originally `int`) follows the official BAAC classification provided.  

It describes the type of road intersection where the accident occurred, including non-intersection segments, standard junctions (e.g., T or X intersections), and roundabouts.  

For modeling purposes, these categories are grouped into broader classes to reflect differences in traffic complexity.

#### 2.4.2 User-Level Data (Usagers)

In [303]:
data["user_type"] = data["user_type"].map({
    1: "driver",
    2: "passenger",
    3: "pedestrian"
})

**Note:** According to the official BAAC classification, the variable `catu` (catégorie d’usager) distinguishes the role of the road user:  
> 1 = driver (conducteur), 2 = passenger (passager), 3 = pedestrian (piéton).

In [304]:
data["is_severe"] = (data["severity"] >= 3).astype(int)

**Note:** 
> Severe injury is defined as grav ≥ 3 (hospitalized or fatal), following the official BAAC(Bulletin d’Analyse des Accidents Corporels) classification.

In [305]:
data["sex"] = data["sex"].map({1: "male", 2: "female"})

In [306]:
data["age"] = 2024 - data["birth_year"]
data = data.drop(columns=["birth_year"])

In [307]:
data["has_safety"] = (
    (data[["secu1", "secu2", "secu3"]] > 0).any(axis=1)
).astype(int)
data = data.drop(columns=["secu1", "secu2", "secu3"])

**Note:** The variables `secu1`, `secu2`, and `secu3` represent the safety equipment used by each road user.  
To simplify the analysis, these variables are combined into a binary feature indicating whether any safety equipment was used.

#### 2.4.3  Vehicle-Level Data (Vehicles)

In [308]:
catv_map = {
    1: "bicycle",
    2: "motorcycle",
    3: "scooter",   # small motorcycle / scooter
    7: "car",
    10: "van",         # light commercial vehicle
    13: "bus",
    14: "truck",
    34: "emergency"
}

data["vehicle_type"] = data["vehicle_type"].map(catv_map).fillna("other")

**Note:** Vehicle types are grouped into interpretable categories (e.g., car, motorcycle, bicycle).  

Light commercial vehicles (catv = 10) are labeled as "van".

In [309]:
def group_impact(x):
    if pd.isna(x) or x == -1:
        return "unknown"
    if x == 0:
        return "none"
    if x in [1, 2, 3]:
        return "front"
    if x in [4, 5, 6]:
        return "rear"
    if x in [7, 8]:
        return "side"
    if x == 9:
        return "multiple"
    return "other"

data["impact"] = data["impact"].apply(group_impact)

**Note:** 

The variable `impact` (originally `choc`) follows the official BAAC classification provided.  

It describes the initial point of impact on the vehicle (e.g., front, rear, side, or multiple impacts).  

For modeling purposes, these categories are grouped into broader classes to capture differences in collision dynamics.

In [310]:
def group_maneuver(x):
    if pd.isna(x) or x in [-1, 0]:
        return "unknown"
    if x in [1, 2, 3]:
        return "straight"
    if x == 4:
        return "reverse"
    if x in [11, 12]:
        return "lane_change"
    if x in [13, 14]:
        return "deviation"
    if x in [15, 16]:
        return "turn"
    if x == 17:
        return "overtake"
    if x in [9, 10]:
        return "merge_uturn"
    return "other"

data["maneuver"] = data["maneuver"].apply(group_maneuver)

In [311]:
data.columns

Index(['accident_id', 'month', 'day', 'time', 'light', 'weather',
       'intersection', 'road', 'user_type', 'vehicle_id', 'user_id',
       'severity', 'sex', 'direction', 'vehicle_type', 'impact', 'maneuver',
       'is_severe', 'age', 'has_safety'],
      dtype='object')

#### 2.4.4 Final Analytical Dataset

In [312]:
df_model = data.drop(columns=[
    "accident_id",
    "vehicle_id",
    "user_id",
    "road",
    "severity",
    "direction",
    "day"
]).copy()

In [313]:
df_model["hour"] = pd.to_datetime(df_model["time"], format="%H:%M", errors="coerce").dt.hour
df_model = df_model.drop(columns=["time"])

In [314]:
ordered_cols = [
    # environment / road conditions
    "month",
    "hour",
    "light",
    "weather",
    "intersection",

    # user-related features
    "user_type",
    "sex",
    "age",
    "has_safety",

    # vehicle-related features
    "vehicle_type",
    "impact",
    "maneuver",

    # target
    "is_severe"
]

df_model = df_model[ordered_cols].copy()

In [315]:
df_model = df_model.dropna()

In [316]:
df_model.head()

,month,hour,light,weather,intersection,user_type,sex,age,has_safety,vehicle_type,impact,maneuver,is_severe
0,3,7,twilight,fog_smoke,no_intersection,driver,male,21.0,1,car,front,deviation,1
1,3,7,twilight,fog_smoke,no_intersection,driver,male,27.0,1,truck,front,other,0
2,3,15,day,wind_storm,standard_intersection,pedestrian,female,97.0,0,van,front,turn,1
3,3,15,day,wind_storm,standard_intersection,driver,male,37.0,1,van,front,turn,0
4,3,19,twilight,clear,no_intersection,passenger,female,17.0,1,scooter,front,straight,1


### 3. Modeling

#### 3.1 Features and Target

In [ ]:
categorical_features = [
    "month",
    "light",
    "weather",
    "intersection",
    "user_type",
    "sex",
    "vehicle_type",
    "impact",
    "maneuver"
]

In [318]:
numeric_features = [
    "hour",
    "age",
    "has_safety"
]

In [319]:
#target
y = df_model["is_severe"]

#### 3.2 Model Training

In [320]:

X = df_model.drop(columns=["is_severe"])
y = df_model["is_severe"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

In [321]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [326]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [323]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'light', 'weather',
                                                   'intersection', 'user_type',
                                                   'sex', 'vehicle_type',
                                                   'impact', 'maneuver']),
                                                 ('num', 'passthrough',
                                                  ['hour', 'age',
                                                   'has_safety'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

#### 3.3 Evaluation and Threshold Adjustment

In [324]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)

In [325]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))


print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6710574609518372
ROC-AUC: 0.7744784024783502
[[ 3881  6874]
 [ 1192 12574]]


## 4. Discussion

The model achieves a ROC-AUC of 0.77, indicating a reasonable ability to distinguish between severe and non-severe cases. After lowering the classification threshold to 0.3, the model becomes more sensitive to severe accidents, significantly reducing false negatives and increasing true positives.

However, this comes at the cost of more false positives, meaning that more non-severe cases are classified as severe. This reflects a trade-off between recall and precision.

In a safety context, this shift can be justified, as missing severe cases is generally more critical than overestimating risk. Overall, the results highlight the importance of threshold selection in classification tasks.

## 5. Conclusion

This project develops a baseline model to predict accident severity using the BAAC dataset. Logistic regression provides moderate performance and captures meaningful patterns in the data.

By adjusting the classification threshold, the model can better detect severe cases, although at the cost of increased false positives. This shows that model evaluation should consider task-specific priorities rather than relying only on accuracy.

Overall, the project demonstrates both the potential and limitations of using structured accident data for severity prediction.